# Duka Electronics: H1 2026 Sales Exploration

**Engagement 1 · Prepared for:** Head of Sales, ahead of Thursday's half-year review (January–June 2026)
**Data:** `duka_sales.csv` (1,000 orders, generated with seed 2026 by `generate_data.py`) · **Currency:** Kenya shillings (KES)

## Executive Summary

- **Revenue is concentrated in Nairobi:** Duka booked KES 71.8M from 1,000 orders in January–June 2026, and Nairobi alone delivered KES 26.1M (36.4%), almost double second-placed Mombasa's KES 14.0M (19.5%).
- **A few high-priced products carry the business:** Premium products (laptops and tablets) are only 43.5% of orders but 84.8% of revenue, and laptops alone earn KES 41.5M (57.9%) even though tablets sell more units (691 vs 639).
- **Weekends behave like weekdays:** the 277 weekend orders (27.7% of the total) are worth KES 19.9M, and the average weekend order (KES 71,972) is within 0.3% of a weekday order (KES 71,724).
- **Sales have slid since April:** KES 13.52M in April was the best month and KES 10.90M in June the worst, so April was 24.0% higher than June (equivalently, June is 19.3% below the peak) after two consecutive monthly declines.
- **A clear target segment exists:** 83 Premium online orders from Nairobi and Mombasa generated KES 10.2M, or 14.2% of all H1 revenue, from just 8.3% of orders.

## 0. Setup and data loading
All imports are in one cell. If `duka_sales.csv` is missing (raw CSVs are deliberately not committed to the repository) the generator is run first, so the notebook can be re-run from a clean clone. Dates are parsed on load with `parse_dates` so that `.dt` works straight away.

In [1]:
import subprocess, sys, zipfile
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

if not Path("duka_sales.csv").exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)

sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
print("Loaded", sales.shape[0], "rows x", sales.shape[1], "columns")

Loaded 1000 rows x 7 columns


## Task 1: First-five-minutes inspection
Before any analysis I check what the data looks like: its size, column types, gaps, the period covered and the key categories.

In [2]:
display(sales.head())
display(sales.tail())
raw_shape = sales.shape
print("shape:", raw_shape)

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


,order_id,order_date,branch,product,quantity,channel,unit_price
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200


shape: (1000, 7)


In [3]:
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[us]
 2   branch      1000 non-null   str           
 3   product     1000 non-null   str           
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   str           
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[us](1), int64(3), str(3)
memory usage: 54.8 KB


In [4]:
sales.describe()

,order_id,order_date,quantity,unit_price
count,1000.000000,1000,1000.000000,1000.000000
mean,1500.500000,2026-03-30 08:44:09.600000,3.012000,23918.300000
min,1001.000000,2026-01-01 00:00:00,1.000000,1200.000000
25%,1250.750000,2026-02-14 00:00:00,2.000000,2500.000000
50%,1500.500000,2026-03-29 00:00:00,3.000000,15000.000000
75%,1750.250000,2026-05-14 00:00:00,4.000000,28000.000000
max,2000.000000,2026-06-29 00:00:00,5.000000,65000.000000
std,288.819436,NaN,1.416991,23908.790546


In [5]:
n_missing = int(sales.isna().sum().sum())
print("Missing values per column:")
print(sales.isna().sum().to_string())
print("\nDuplicated order_id:", sales["order_id"].duplicated().sum())
print("\nDistinct values per column:")
print(sales.nunique().to_string())

Missing values per column:
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0

Duplicated order_id: 0

Distinct values per column:
order_id      1000
order_date     179
branch           5
product          5
quantity         5
channel          3
unit_price       5


In [6]:
first, last = sales["order_date"].min(), sales["order_date"].max()
print(f"Date range: {first:%Y-%m-%d} to {last:%Y-%m-%d} ({(last - first).days + 1} calendar days)")
print("\nOrders per calendar month:")
print(sales["order_date"].dt.to_period("M").value_counts().sort_index().to_string())

Date range: 2026-01-01 to 2026-06-29 (180 calendar days)

Orders per calendar month:
order_date
2026-01    173
2026-02    159
2026-03    186
2026-04    167
2026-05    153
2026-06    162
Freq: M


In [7]:
for col in ["branch", "channel", "product"]:
    print(f"--- {col} (% of orders) ---")
    print((sales[col].value_counts(normalize=True) * 100).round(1).to_string(), "\n")
print("--- unit_price by product (KES) ---")
print(sales.groupby("product")["unit_price"].agg(["min", "max"]).to_string())

--- branch (% of orders) ---
branch
Nairobi    37.6
Mombasa    18.6
Nakuru     16.2
Kisumu     15.9
Eldoret    11.7 

--- channel (% of orders) ---
channel
In-store       57.9
Online         32.0
Phone order    10.1 

--- product (% of orders) ---
product
Laptop        22.0
Tablet        21.5
Phone         19.4
Headphones    18.7
Charger       18.4 

--- unit_price by product (KES) ---
              min    max
product                 
Charger      1200   1200
Headphones   2500   2500
Laptop      65000  65000
Phone       15000  15000
Tablet      28000  28000


### What the inspection shows
- **Shape:** 1,000 rows × 7 columns, with one row per order (`order_id` runs 1001–2000 with no duplicates).
- **Types:** `order_date` was parsed as `datetime64` (not text); `quantity` and `unit_price` are integers; `branch`, `product` and `channel` are text categories.
- **Missing values:** 0 in any column, so no imputation or row-dropping is needed.
- **Date range:** 01 Jan 2026 to 29 Jun 2026. The generator draws days 0–179 from 1 January, so **30 June never appears**: June has 29 days of data, one fewer than a full month. I account for this when comparing months in Task 6.
- **Key categories:** 5 branches (Nairobi 37.6% of orders, then Mombasa 18.6%, Nakuru 16.2%, Kisumu 15.9%, Eldoret 11.7%); 5 products (184–220 orders each, so fairly evenly spread); 3 channels (In-store 57.9%, Online 32.0%, Phone order 10.1%).
- **Ranges:** `quantity` is 1–5 per order; `unit_price` takes exactly one value per product (KES 1,200 for chargers up to KES 65,000 for laptops), so every order's revenue is fully determined by product and quantity.

## Task 2: Derived columns
`revenue = quantity × unit_price`; `month` is a monthly period (sorts chronologically); `weekday` is the day name; `is_weekend` is Saturday/Sunday; `price_band` is **Premium** when `unit_price` ≥ KES 28,000 and **Value** otherwise (the Tablet, at exactly KES 28,000, is therefore Premium).

In [8]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.to_period("M")
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(sales["unit_price"] >= 28_000, "Premium", "Value")

sales[["order_id", "order_date", "product", "quantity", "unit_price",
       "revenue", "month", "weekday", "is_weekend", "price_band"]].head(8)

,order_id,order_date,product,quantity,unit_price,revenue,month,weekday,is_weekend,price_band
0,1001,2026-06-03,Phone,3,15000,45000,2026-06,Wednesday,False,Value
1,1002,2026-02-02,Laptop,4,65000,260000,2026-02,Monday,False,Premium
2,1003,2026-01-05,Tablet,5,28000,140000,2026-01,Monday,False,Premium
3,1004,2026-04-26,Headphones,5,2500,12500,2026-04,Sunday,True,Value
4,1005,2026-03-07,Headphones,3,2500,7500,2026-03,Saturday,True,Value
5,1006,2026-03-26,Headphones,1,2500,2500,2026-03,Thursday,False,Value
6,1007,2026-01-15,Laptop,2,65000,130000,2026-01,Thursday,False,Premium
7,1008,2026-03-08,Charger,5,1200,6000,2026-03,Sunday,True,Value


In [9]:
# Sanity checks on the new columns
assert sales["revenue"].min() > 0
assert sales["month"].nunique() == 6
assert set(sales["price_band"]) == {"Premium", "Value"}
assert (sales.loc[sales["price_band"] == "Premium", "unit_price"] >= 28_000).all()
assert (sales.loc[sales["price_band"] == "Value", "unit_price"] < 28_000).all()

total_revenue = sales["revenue"].sum()
n_orders = len(sales)
avg_order = total_revenue / n_orders
print(f"Total revenue: KES {total_revenue:,}   Orders: {n_orders:,}   Average order: KES {avg_order:,.0f}")
print()
print(sales.groupby("price_band").agg(products=("product", lambda x: ", ".join(sorted(x.unique()))),
                                      orders=("order_id", "count"), revenue=("revenue", "sum")).to_string())
print()
print(sales["is_weekend"].value_counts().rename({False: "weekday", True: "weekend"}).to_string())

Total revenue: KES 71,792,500   Orders: 1,000   Average order: KES 71,792

                              products  orders   revenue
price_band                                              
Premium                 Laptop, Tablet     435  60883000
Value       Charger, Headphones, Phone     565  10909500

is_weekend
weekday    723
weekend    277


## Task 3: Branch ranking by revenue and by number of orders

In [10]:
branch = sales.groupby("branch").agg(revenue=("revenue", "sum"), orders=("order_id", "count"), units=("quantity", "sum"))
branch["revenue_share_%"] = (branch["revenue"] / branch["revenue"].sum() * 100).round(1)
branch["order_share_%"] = (branch["orders"] / branch["orders"].sum() * 100).round(1)
branch["avg_order_value"] = (branch["revenue"] / branch["orders"]).round(0)
branch["rank_revenue"] = branch["revenue"].rank(ascending=False, method="min").astype(int)
branch["rank_orders"] = branch["orders"].rank(ascending=False, method="min").astype(int)
branch = branch.sort_values("revenue", ascending=False)
display(branch)

print("Rankings identical?", (branch["rank_revenue"] == branch["rank_orders"]).all())

# Is the product mix similar in every branch? (% of each branch's orders by price band)
mix = (pd.crosstab(sales["branch"], sales["price_band"], normalize="index") * 100).round(1)
display(mix.loc[branch.index])

,revenue,orders,units,revenue_share_%,order_share_%,avg_order_value,rank_revenue,rank_orders
branch,,,,,,,,
Nairobi,26106100,376,1107,36.4,37.6,69431.0,1,1
Mombasa,14003700,186,567,19.5,18.6,75289.0,2,2
Nakuru,11970400,162,495,16.7,16.2,73891.0,3,3
Kisumu,10868100,159,473,15.1,15.9,68353.0,4,4
Eldoret,8844200,117,370,12.3,11.7,75591.0,5,5


Rankings identical? True


price_band,Premium,Value
branch,,
Nairobi,43.9,56.1
Mombasa,50.0,50.0
Nakuru,43.2,56.8
Kisumu,41.5,58.5
Eldoret,35.0,65.0


### Answer
**Yes, the two rankings are identical:** Nairobi (KES 26.1M, 376 orders) > Mombasa (KES 14.0M, 186 orders) > Nakuru (KES 12.0M, 162 orders) > Kisumu (KES 10.9M, 159 orders) > Eldoret (KES 8.8M, 117 orders).

**Why they agree.** Revenue = orders × average order value (AOV), and the two ingredients vary very differently across branches. Order counts differ enormously (376 in Nairobi vs 117 in Eldoret, a 3.2× spread), while AOV sits in a narrow band, from KES 68,353 to KES 75,591 (a 1.11× spread, never more than 5.3% from the company average of KES 71,793). Revenue therefore follows order volume (3.0× spread), and a branch with more orders also has more revenue. AOV differences are too small to reorder branches that are as far apart in volume as these.

**What the small differences tell us**
- **Nairobi leads on volume, not value.** It has 37.6% of orders but 36.4% of revenue, and its AOV (KES 69,431) is *below* the company average.
- **Nakuru vs Kisumu is the one place where order value matters.** They are nearly tied on orders (162 vs 159, +1.9%), yet Nakuru earns 10.1% more revenue because its AOV is 8.1% higher. That was not enough to change the order of the ranking.
- **Eldoret has the highest AOV (KES 75,591) but the fewest orders (117)**, so it ranks last on both measures. Its opportunity is order volume, not basket size.

## Task 4: Top three products by revenue and by units sold

In [11]:
product = sales.groupby("product").agg(revenue=("revenue", "sum"), units=("quantity", "sum"),
                                       orders=("order_id", "count"), unit_price=("unit_price", "first"))
product["revenue_share_%"] = (product["revenue"] / product["revenue"].sum() * 100).round(1)
product["unit_share_%"] = (product["units"] / product["units"].sum() * 100).round(1)

top_rev = product.nlargest(3, "revenue")
top_units = product.nlargest(3, "units")
print("Top 3 by REVENUE"); display(top_rev[["revenue", "revenue_share_%", "units", "unit_price"]])
print("Top 3 by UNITS");   display(top_units[["units", "unit_share_%", "revenue", "unit_price"]])
print("Same three products?", set(top_rev.index) == set(top_units.index),
      "| same order?", list(top_rev.index) == list(top_units.index))
display(product.sort_values("revenue", ascending=False))

Top 3 by REVENUE


,revenue,revenue_share_%,units,unit_price
product,,,,
Laptop,41535000,57.9,639,65000
Tablet,19348000,26.9,691,28000
Phone,8880000,12.4,592,15000


Top 3 by UNITS


,units,unit_share_%,revenue,unit_price
product,,,,
Tablet,691,22.9,19348000,28000
Laptop,639,21.2,41535000,65000
Phone,592,19.7,8880000,15000


Same three products? True | same order? False


,revenue,units,orders,unit_price,revenue_share_%,unit_share_%
product,,,,,,
Laptop,41535000,639,220,65000,57.9,21.2
Tablet,19348000,691,215,28000,26.9,22.9
Phone,8880000,592,194,15000,12.4,19.7
Headphones,1387500,555,187,2500,1.9,18.4
Charger,642000,535,184,1200,0.9,17.8


### Answer
| Rank | By revenue | By units sold |
|---|---|---|
| 1 | **Laptop**: KES 41.5M (57.9%) | **Tablet**: 691 units |
| 2 | **Tablet**: KES 19.3M (26.9%) | **Laptop**: 639 units |
| 3 | **Phone**: KES 8.9M (12.4%) | **Phone**: 592 units |

**Why the lists differ.** They contain the same three products; only the top two swap places. Revenue is units × price, and the two products sit at opposite ends of that trade-off: Tablets sold 52 more units than Laptops (8.1% more), but a Laptop costs KES 65,000 against KES 28,000 for a Tablet, so Laptops earn 2.15× the revenue.

Unit sales are almost flat across the range (every product is between 17.8% and 22.9% of units), so the *units* ranking is close to a tie. Price is what separates products. Headphones and chargers make up 36.2% of units but only 2.8% of revenue.

**So what:** rank products by revenue when deciding where to focus stock, promotion and sales effort (Laptops and Tablets), and use units only to size logistics and accessory restocking.

## Task 5: Weekend vs weekday orders

In [12]:
we = sales.groupby("is_weekend").agg(orders=("order_id", "count"), revenue=("revenue", "sum"),
                                     avg_order_revenue=("revenue", "mean"))
we.index = we.index.map({False: "Weekday", True: "Weekend"})
we["order_share_%"] = (we["orders"] / we["orders"].sum() * 100).round(1)
we["revenue_share_%"] = (we["revenue"] / we["revenue"].sum() * 100).round(1)

# Fair comparison: weekdays and weekends do not have the same number of days in the period
calendar = pd.Series(pd.date_range(sales["order_date"].min(), sales["order_date"].max()))
we["calendar_days"] = calendar.dt.dayofweek.ge(5).map({False: "Weekday", True: "Weekend"}).value_counts()
we["orders_per_day"] = (we["orders"] / we["calendar_days"]).round(2)
we["revenue_per_day"] = (we["revenue"] / we["calendar_days"]).round(0)
we["avg_order_revenue"] = we["avg_order_revenue"].round(0)
we

,orders,revenue,avg_order_revenue,order_share_%,revenue_share_%,calendar_days,orders_per_day,revenue_per_day
is_weekend,,,,,,,,
Weekday,723,51856200,71724.0,72.3,72.2,128,5.65,405127.0
Weekend,277,19936300,71972.0,27.7,27.8,52,5.33,383390.0


### Answer
| | Orders | Total revenue | Average order revenue |
|---|---|---|---|
| **Weekday** (Mon–Fri) | 723 | KES 51,856,200 | KES 71,724 |
| **Weekend** (Sat–Sun) | 277 | KES 19,936,300 | KES 71,972 |

Weekdays produce 72.2% of revenue and weekends 27.8%. Weekend days are 28.9% of the calendar (52 of 180 days), so weekends deliver slightly *less* than their calendar share. Per day, weekends are slightly *quieter*: 5.33 orders and KES 383,390 a day, against 5.65 orders and KES 405,127 on a weekday (-5.4% on revenue). The average order is essentially the same (+0.3%), so weekend orders are not bigger than weekday ones.

**So what:** weekends are not a special sales opportunity at Duka; they deliver a normal share of trade. A weekend-only promotion would have to *create* demand rather than ride existing demand.

## Task 6: Best and worst month for revenue

In [13]:
monthly = sales.groupby("month").agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
days_idx = pd.date_range(sales["order_date"].min(), sales["order_date"].max())
monthly["days_covered"] = pd.Series(1, index=days_idx).groupby(days_idx.to_period("M")).sum()
monthly["revenue_per_day"] = (monthly["revenue"] / monthly["days_covered"]).round(0)
monthly["avg_order_value"] = (monthly["revenue"] / monthly["orders"]).round(0)
monthly["change_vs_prev_month_%"] = (monthly["revenue"].pct_change() * 100).round(1)

best, worst = monthly["revenue"].idxmax(), monthly["revenue"].idxmin()
pct_best_over_worst = (monthly["revenue"].max() - monthly["revenue"].min()) / monthly["revenue"].min() * 100
pct_worst_below_best = (monthly["revenue"].max() - monthly["revenue"].min()) / monthly["revenue"].max() * 100
display(monthly)
print(f"Best month : {best}  KES {monthly.loc[best, 'revenue']:,}")
print(f"Worst month: {worst}  KES {monthly.loc[worst, 'revenue']:,}")
print(f"Best is {pct_best_over_worst:.1f}% higher than worst; worst is {pct_worst_below_best:.1f}% lower than best")

,revenue,orders,days_covered,revenue_per_day,avg_order_value,change_vs_prev_month_%
month,,,,,,
2026-01,11786600,173,31,380213.0,68131.0,NaN
2026-02,11772000,159,28,420429.0,74038.0,-0.1
2026-03,12396800,186,31,399897.0,66649.0,5.3
2026-04,13518300,167,30,450610.0,80948.0,9.0
2026-05,11415400,153,31,368239.0,74610.0,-15.6
2026-06,10903400,162,29,375979.0,67305.0,-4.5


Best month : 2026-04  KES 13,518,300
Worst month: 2026-06  KES 10,903,400
Best is 24.0% higher than worst; worst is 19.3% lower than best


### Answer
- **Best month: April 2026** with KES 13,518,300.
- **Worst month: June 2026** with KES 10,903,400.
- **Percentage difference:** (KES 13,518,300 − KES 10,903,400) ÷ KES 10,903,400 = **24.0%**: the best month is 24.0% above the worst. Measured the other way, the worst month is 19.3% below the best (always state the base you divide by).

**Is the gap real or a calendar effect?** The dataset covers only 29 days of June, so I checked revenue *per day covered*: April earned KES 450,610 a day against KES 375,979 in June, a 19.8% gap. That is smaller than the headline figure but April is still clearly the peak and June the trough, so the conclusion holds.

**Trend.** After the April peak revenue fell 15.6% in May and a further 4.5% in June; the half-year ended on its lowest month. Order *counts* did not collapse the same way (167 orders in April, 162 in June), and average order value fell from KES 80,948 to KES 67,305, so the decline is mostly smaller baskets rather than fewer orders: a question worth putting to the sales team on Thursday.

## Task 7: Premium online orders from Nairobi and Mombasa

In [14]:
mask = (
    (sales["price_band"] == "Premium")
    & (sales["channel"] == "Online")
    & (sales["branch"].isin(["Nairobi", "Mombasa"]))
)
premium_online = sales[mask].copy()

n_po, rev_po = len(premium_online), premium_online["revenue"].sum()
print(f"Premium online orders (Nairobi + Mombasa): {n_po}")
print(f"Total revenue: KES {rev_po:,}")
print(f"Share of all orders : {n_po / len(sales) * 100:.1f}%")
print(f"Share of all revenue: {rev_po / sales['revenue'].sum() * 100:.1f}%")
print(f"Share of Premium revenue: {rev_po / sales.loc[sales['price_band'] == 'Premium', 'revenue'].sum() * 100:.1f}%")

display(premium_online.groupby(["branch", "product"]).agg(orders=("order_id", "count"), revenue=("revenue", "sum")))

Path("outputs").mkdir(exist_ok=True)
premium_online.to_csv("outputs/premium_online_coast_capital.csv", index=False)

# Verify the exported file
check = pd.read_csv("outputs/premium_online_coast_capital.csv")
assert len(check) == n_po and check["revenue"].sum() == rev_po
assert set(check["price_band"]) == {"Premium"} and set(check["channel"]) == {"Online"}
assert set(check["branch"]) == {"Nairobi", "Mombasa"} and "Unnamed: 0" not in check.columns
print("\nExported outputs/premium_online_coast_capital.csv:", check.shape, "(index=False, filters verified)")

Premium online orders (Nairobi + Mombasa): 83
Total revenue: KES 10,218,000
Share of all orders : 8.3%
Share of all revenue: 14.2%
Share of Premium revenue: 16.8%


orders  revenue
branch  product                 
Mombasa Laptop       10  1820000
        Tablet       23  1876000
Nairobi Laptop       23  4030000
        Tablet       27  2492000


Exported outputs/premium_online_coast_capital.csv: (83, 12) (index=False, filters verified)


### Answer
There are **83 Premium online orders** from Nairobi and Mombasa, with a total revenue of **KES 10,218,000**. The file is saved as `outputs/premium_online_coast_capital.csv` (`index=False`).

- **Size of the segment:** 8.3% of all orders but 14.2% of total revenue. That is 16.8% of all Premium revenue.
- **Split:** Nairobi 50 orders (KES 6.5M) and Mombasa 33 orders (KES 3.7M).
- **Average order:** KES 123,108 against KES 71,793 for all orders, so each of these orders is worth about 1.7× a typical one. It is a small, high-value segment that merits its own online offer for laptops and tablets.

## Task 8: Spreadsheet reconciliation (`excel/duka_branch_summary.xlsx`)
The workbook has four sheets: **`data`** (the 1,000 orders with `revenue = E×G` as a live formula), **`branch_summary`** (branch revenue with `SUMIFS`, order counts with `COUNTIFS`, share, `RANK`, and a `region` column using `XLOOKUP` against the **`regions`** sheet), and **`pivot_branch_channel`** (a PivotTable of revenue by branch × channel). The cell below reads the values stored in the workbook and compares them with pandas.

In [15]:
xl_path = "excel/duka_branch_summary.xlsx"

# 1) SUMIFS branch table vs pandas
xl_branch = pd.read_excel(xl_path, sheet_name="branch_summary", nrows=5).set_index("branch")
pd_branch = sales.groupby("branch")["revenue"].sum()
cmp = pd.DataFrame({"excel_SUMIFS": xl_branch["total_revenue_kes"], "pandas": pd_branch}).dropna()
cmp["difference"] = cmp["excel_SUMIFS"] - cmp["pandas"]
display(cmp.join(xl_branch[["region"]]))
assert len(cmp) == 5 and (cmp["difference"] == 0).all()

# 2) PivotTable branch x channel vs pandas
xl_pivot = pd.read_excel(xl_path, sheet_name="pivot_branch_channel", header=3, index_col=0).dropna(how="all")
pd_pivot = sales.pivot_table(index="branch", columns="channel", values="revenue", aggfunc="sum", margins=True, margins_name="Grand Total")
pd_pivot = pd_pivot.rename(columns={"Grand Total": "Grand Total"})
assert (xl_pivot.astype(float).values == pd_pivot.reindex(index=xl_pivot.index, columns=xl_pivot.columns).values).all()

print("Workbook grand total :", f"KES {int(xl_pivot.loc['Grand Total', 'Grand Total']):,}")
print("pandas total revenue :", f"KES {int(sales['revenue'].sum()):,}")
print("SUMIFS table total   :", f"KES {int(xl_branch['total_revenue_kes'].sum()):,}")
assert int(xl_pivot.loc["Grand Total", "Grand Total"]) == int(sales["revenue"].sum()) == int(xl_branch["total_revenue_kes"].sum())

# 3) The workbook really contains formulas and a PivotTable (not pasted values)
import openpyxl
wb_f = openpyxl.load_workbook(xl_path)
print("\nB2 :", wb_f["branch_summary"]["B2"].value)
print("F2 :", wb_f["branch_summary"]["F2"].value)
print("data!H2 :", wb_f["data"]["H2"].value)
print("PivotTable parts in file:", [n for n in zipfile.ZipFile(xl_path).namelist() if "pivotTable" in n and n.endswith(".xml") and "_rels" not in n])
print("\n>>> Excel and pandas totals match exactly.")
display(xl_pivot)

,excel_SUMIFS,pandas,difference,region
branch,,,,
Eldoret,8844200,8844200,0,Rift Valley
Kisumu,10868100,10868100,0,Lake (Nyanza)
Mombasa,14003700,14003700,0,Coast
Nairobi,26106100,26106100,0,Nairobi
Nakuru,11970400,11970400,0,Rift Valley


Workbook grand total : KES 71,792,500
pandas total revenue : KES 71,792,500
SUMIFS table total   : KES 71,792,500

B2 : =SUMIFS(data!$H$2:$H$1001,data!$C$2:$C$1001,A2)
F2 : =_xlfn.XLOOKUP(A2,regions!$A$2:$A$6,regions!$B$2:$B$6,"Not found")
data!H2 : =E2*G2
PivotTable parts in file: ['xl/pivotTables/pivotTable1.xml']

>>> Excel and pandas totals match exactly.


,In-store,Online,Phone order,Grand Total
Row Labels,,,,
Eldoret,4885300,3450700,508200,8844200
Kisumu,6468600,3370800,1028700,10868100
Mombasa,8039600,4086800,1877300,14003700
Nairobi,14808400,7527400,3770300,26106100
Nakuru,6767300,4350400,852700,11970400
Grand Total,40969200,22786100,8037200,71792500


### Answer
The SUMIFS branch totals, the `COUNTIFS` order counts and the PivotTable all reconcile to pandas **to the shilling** (differences are all zero, asserted in the cell above), and the workbook grand total equals the pandas total. Two notes for whoever opens the file:
- `region` is **not in the Duka data**; the branch → region mapping on the `regions` sheet (Nairobi → Nairobi, Mombasa → Coast, Kisumu → Lake (Nyanza), Nakuru and Eldoret → Rift Valley) is my assumption and should be confirmed by Sales.
- `XLOOKUP` needs Excel 2021/Microsoft 365 or Google Sheets (older Excel and LibreOffice < 24.8 show `#NAME?`). The PivotTable is set to refresh when the file is opened; if Excel asks, choose *Enable*.

## Conclusions
The Executive Summary at the top of this notebook is the Task 9 deliverable. Recommended follow-ups for the review meeting:

1. **Treat April–June as the headline problem.** Revenue is down 19.3% from the April peak and average order value is falling, so ask which products and branches lost basket size.
2. **Protect Nairobi, grow value elsewhere.** Nairobi (36.4% of revenue) wins on volume; the other branches' growth lever is more orders (Eldoret) or larger baskets (Kisumu, Nairobi).
3. **Plan around revenue, not units.** Laptops and Tablets drive the business; accessories sell in volume but add under 3% of revenue.
4. **Target Premium online buyers in Nairobi and Mombasa** (83 orders, KES 10.2M) with a focused offer rather than a weekend-only promotion, since weekends show no special demand.

*Caveats: the data covers 1 January–29 June 2026 (no 30 June), and the branch → region mapping in the workbook is an assumption.*